In [ ]:
!pip install flask
import numpy as np
import librosa
import tensorflow as tf
from flask import Flask, request, jsonify
import os


MODEL_PATH = "language_cnn.keras"
CLASS_PATH = "language_classes.npy"

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 200

# Load model ONCE
model = tf.keras.models.load_model(MODEL_PATH)

# Load language names
classes = np.load(
    CLASS_PATH,
    allow_pickle=True
)


def extract_mfcc(file_path):

    audio, sr = librosa.load(
        file_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    # Normalize
    audio = librosa.util.normalize(audio)

    # Trim silence
    audio, _ = librosa.effects.trim(
        audio,
        top_db=20
    )

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC
    )

    # Fixed length
    if mfcc.shape[1] < MAX_LEN:

        pad_width = MAX_LEN - mfcc.shape[1]

        mfcc = np.pad(
            mfcc,
            ((0, 0), (0, pad_width)),
            mode="constant"
        )

    else:

        mfcc = mfcc[:, :MAX_LEN]

    return mfcc


def predict_language(file_path):

    mfcc = extract_mfcc(file_path)

    # Shape:
    # (40, 200)
    print("MFCC:", mfcc.shape)

    # Add channel dimension
    mfcc = mfcc[..., np.newaxis]

    # Add batch dimension
    mfcc = np.expand_dims(
        mfcc,
        axis=0
    )

    # Prediction
    probabilities = model.predict(
        mfcc,
        verbose=0
    )[0]

    # Highest probability
    index = np.argmax(probabilities)

    language = classes[index]

    confidence = probabilities[index] * 100

    return language, confidence, probabilities

    


app = Flask(__name__)

UPLOAD_FOLDER = "uploads"

os.makedirs(
    UPLOAD_FOLDER,
    exist_ok=True
)


@app.route("/predict", methods=["POST"])
def predict():

    if "audio" not in request.files:

        return jsonify({
            "error": "No audio file received"
        }), 400

    audio = request.files["audio"]

    file_path = os.path.join(
        UPLOAD_FOLDER,
        audio.filename
    )

    audio.save(file_path)

    language, confidence, probabilities = \
        predict_language(file_path)

    result = {

        "language": language,

        "confidence": round(
            float(confidence),
            2
        ),

        "probabilities": {

            "Dogri": round(
                float(probabilities[0]) * 100,
                2
            ),

            "English": round(
                float(probabilities[1]) * 100,
                2
            ),

            "Hindi": round(
                float(probabilities[2]) * 100,
                2
            )
        }
    }

    return jsonify(result)


if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False
   
    )

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Oct/2026 13:05:31] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [05/Oct/2026 13:05:31] "GET /favicon.ico HTTP/1.1" 404 -
